# Ir Além 2 - Diagnóstico Visual em Cardiologia com Rede Neural (MLP)

Neste notebook, construímos uma Rede Neural Artificial (MLP - Multilayer Perceptron) utilizando a biblioteca **Keras / TensorFlow** para identificar de forma automatizada anomalias no ritmo cardíaco através da análise de sinais de Eletrocardiogramas (ECG).

Utilizamos a base pública sugerida do Kaggle (*Heartbeat*), especificamente a divisão **PTB Diagnostic ECG Database**, pois ela é perfeita para o problema solicitado de **Classificação Binária** (Normal vs. Anormal).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

# Configuração Padrão
import warnings
warnings.filterwarnings('ignore')

### 1. Pré-processamento e Carregamento dos Dados
Importamos as vertentes do dataset. Vale ressaltar que os dados neste Kaggle são equivalentes a imagens de sinais em tons de cinza/intensidade que já vêm limpos e "achatados" (flatten) em arrays 1D contendo valores entre `0` e `1`. Por esse formato, um MLP é super adequado.

Cada linha representa **1 batimento (beat)** contendo 187 features numéricas. A última coluna traz o *Label*.

In [ ]:
df_normal = pd.read_csv('assets/ptbdb_normal.csv', header=None)
df_abnormal = pd.read_csv('assets/ptbdb_abnormal.csv', header=None)

print(f"Sinais Normais: {df_normal.shape}")
print(f"Sinais Anormais: {df_abnormal.shape}")

### 2. Conversão Visual das Features em Imagem (Para o Entregável)
Vamos recriar a representação do eletrocardiograma observando a curva pura formatada no array e exportar essa visualização convertida como um de nossos artefatos (`exemplo_ecg.png`).

In [ ]:
plt.figure(figsize=(10,4))
plt.plot(df_normal.iloc[0, :187], label="Normal (0)", color="#0c82f0")
plt.plot(df_abnormal.iloc[0, :187], label="Anormal (1)", color="#EF4444")
plt.title("Comparação Visual do Sinal de ECG (Array -> Imagem)")
plt.xlabel("Tempo/Sample")
plt.ylabel("Amplitude Normalizada")
plt.legend()

plt.savefig("exemplo_ecg.png") # Artefato Imagem gerado no diretório
plt.show()

### 3. Preparação dos Vetores de Treinamento e Teste
Mesclamos os dados, e utilizamos o `train_test_split` dividindo nosso conhecimento para que o modelo prove sua capacidade de "ver" dados inéditos após treinado.

In [ ]:
df_total = pd.concat([df_normal, df_abnormal], axis=0, ignore_index=True)

# 187 características de sinal vs Label.
X = df_total.iloc[:, :-1].values
y = df_total.iloc[:, -1].values

# Setando 20% para a validação real de avaliação
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

### 4. Construindo a Arquitetura da Rede Neural (MLP via Keras)
Criaremos uma rede robusta profunda e totalmente conectada (`Dense`)
- `Input = 187 unidades` numéricas do sinal.
- **Dropout** para impedir que haja *Overfitting* (a máquina decorar respostas invés de aprender).
- A **camada de saída final ativa 1 neurônio via Sigmoid** para responder probabilisticamente entre `0` (Normal) e `1` (Anormal).

In [ ]:
modelo_mlp = Sequential()

# Camada de Entrada / Camada Oculta 1
modelo_mlp.add(Dense(128, activation='relu', input_shape=(187,)))
modelo_mlp.add(Dropout(0.2))

# Camada Oculta 2
modelo_mlp.add(Dense(64, activation='relu'))
modelo_mlp.add(Dropout(0.1))

# Camada Oculta 3
modelo_mlp.add(Dense(32, activation='relu'))

# Camada de Saída (Classificação Binária)
modelo_mlp.add(Dense(1, activation='sigmoid'))

# Compilando o compilador de perda binária (Binary Crossentropy)
modelo_mlp.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

modelo_mlp.summary()

### 5. Execução do Treinamento

In [ ]:
print("Iniciando Módulo de Treinamento de IA...")
history = modelo_mlp.fit(X_train, y_train, epochs=20, batch_size=64, validation_split=0.2, verbose=1)

### 6. Avaliação Real da Acurácia no Teste

In [ ]:
# Avaliando
loss, accuracy = modelo_mlp.evaluate(X_test, y_test)
print(f"\nAcurácia do Modelo no Conjunto de Testes: {accuracy*100:.2f}%")

# Representação visual do crescimento do estudo da IA
plt.figure(figsize=(8,5))
plt.plot(history.history['accuracy'], label='Acurácia no Treino')
plt.plot(history.history['val_accuracy'], label='Acurácia na Validação')
plt.title('Taxa de Acertos por Oitava/Época')
plt.xlabel('Época de Treino')
plt.ylabel('Acurácia %')
plt.legend()
plt.show()